# Phase 6 — Retrieval function

The actual retrieval function, built on the decisions made in `scripts/05. search_method_evaluation.ipynb`: hybrid search via RRF (`k=5`, `w=0.85` favoring semantic), semantic side is Pinecone (Cohere `embed-v4.0`), keyword side is BM25 (`k1=1.5`, `b=0.75`, both left at their empirically-confirmed-optimal defaults).

The implementation lives in `src/retrieval.py` (`Retriever` class) so it's importable from a future FastAPI backend (Phase 9), not locked inside a notebook cell. This notebook imports it, demos it on a couple of real questions, and — importantly — validates that it reproduces the recall/MRR numbers from the eval notebook.

**Why validation matters here**: the eval swept RRF over *full-corpus* ranks (every one of the 739 chunks ranked by each method). The real function only pulls each ranker's top 50 candidates (`CANDIDATE_TOP_K`) before fusing, since that's how Pinecone/BM25 actually get queried in production. 50 is a generous margin above the recall@10 cutoff the eval cared about, so the two should match — but that's an assumption worth checking, not assuming.

Kernel: `rag311`.

In [1]:
import sys
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever

retriever = Retriever()
print("chunks loaded:", len(retriever.chunks))

chunks loaded: 739


## Quick demo

In [2]:
demo_questions = [
    "كم عدد أيام الإجازة السنوية التي يستحقها الموظف؟",
    "What compensation am I entitled to if I'm terminated without notice?",
]
for q in demo_questions:
    print(f"\nQ: {q}")
    for r in retriever.retrieve(q, top_k=5):
        clause_suffix = f".{r['clause']}" if r["clause"] else ""
        print(f"  [{r['score']:.4f}] {r['law']} art.{r['article_number']}{clause_suffix} ({r['language']})")


Q: كم عدد أيام الإجازة السنوية التي يستحقها الموظف؟


  [0.1667] labor96 art.61 (ar)
  [0.1277] labor96 art.63 (ar)
  [0.1214] labor96 art.61 (en)
  [0.1038] labor96 art.65 (ar)
  [0.0975] labor96 art.60 (ar)

Q: What compensation am I entitled to if I'm terminated without notice?


  [0.1567] labor96 art.23 (en)
  [0.1293] labor96 art.25 (en)
  [0.1159] labor96 art.28 (en)
  [0.1062] labor96 art.32 (en)
  [0.0897] labor96 art.26 (en)


## Validation against the eval methodology

Run `retriever.retrieve(question, top_k=10)` for all 40 eval questions and recompute recall@1/5/10 + MRR the same way `05. search_method_evaluation.ipynb` did. Expect to reproduce the RRF k=5/w=0.85 numbers from that notebook: recall@1 0.675, recall@5 0.825, recall@10 0.900, MRR 0.744.

In [3]:
import json
import pandas as pd

EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

LAW_MAP = {
    "Work_Law_1996": "labor96",
    "Social_Security_Law": "ssl",
    "Professional_Work_Law_2019": "prof19",
}
eval_questions["answer_law_code"] = eval_questions["answer_law"].map(LAW_MAP)
eval_questions["is_cross_lingual"] = eval_questions["question_language"] != eval_questions["answer_language"]

rows = []
for row in eval_questions.itertuples(index=False):
    correct_articles = set(row.answer_article_numbers)
    results = retriever.retrieve(row.question_text, top_k=10)
    rank = None
    for pos, r in enumerate(results, start=1):
        if r["law"] == row.answer_law_code and r["language"] == row.answer_language and r["article_number"] in correct_articles:
            rank = pos
            break
    rows.append({"question_id": row.question_id, "is_cross_lingual": row.is_cross_lingual, "rank": rank})

details = pd.DataFrame(rows)

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import metrics

for label, subset in [("overall", details), ("same-language", details[~details["is_cross_lingual"]]), ("cross-lingual", details[details["is_cross_lingual"]])]:
    print(label, metrics(subset))

overall {'n': 40, 'recall@1': np.float64(0.675), 'recall@5': np.float64(0.825), 'recall@10': np.float64(0.9), 'mrr': np.float64(0.744047619047619)}
same-language {'n': 32, 'recall@1': np.float64(0.84375), 'recall@5': np.float64(0.9375), 'recall@10': np.float64(0.96875), 'mrr': np.float64(0.8742559523809523)}
cross-lingual {'n': 8, 'recall@1': np.float64(0.0), 'recall@5': np.float64(0.375), 'recall@10': np.float64(0.625), 'mrr': np.float64(0.2232142857142857)}


## Result\n\nExact match with the eval notebook's RRF k=5/w=0.85 numbers -- recall@1 0.675, recall@5 0.825, recall@10 0.900, MRR 0.744. Confirms `CANDIDATE_TOP_K=50` in `src/retrieval.py` is a wide enough net that capping each ranker's candidate list before RRF-fusing doesn't lose anything the full-corpus eval found.\n\nPhase 6 done. `Retriever.retrieve(query, top_k)` in `src/retrieval.py` is the reusable function Phase 7 (generation) will call -- see `ROADMAP.md`.